In [27]:
from pyspark.sql import *
from pyspark.sql.functions import col,lower,to_date,trim


output_path = "C:\\Users\\kumaa\\Downloads\\dirty_dataset.json"

spark=SparkSession.builder\
      .appName("dirty_data_cleansing")\
      .master("local[3]")\
      .getOrCreate()

df=spark.read.json(output_path)
df.show()


+-------+-------+-------------+----------+-------------+------+-----------------+-------+-----------------+--------+
|   city|country|     metadata|product_id|purchase_date|rating|           review|user_id|verified_purchase|zip_code|
+-------+-------+-------------+----------+-------------+------+-----------------+-------+-----------------+--------+
|    n/a|  India|  {NULL, 999}|   000-xyz|   2021-11-11|     5|      Best ever!!|  user4|            maybe|      NA|
|    n/a|   NULL|{value, NULL}|   123-abc|   2021-13-01|     5|  great product  |  USER1|              yes|        |
|    n/a|   NULL|{value, NULL}|   123-abc|   2021-13-01|     5|  great product  |  USER1|              yes|        |
|     uk|     UK| {NULL, NULL}|   789-ghi|   2022-05-20|     2|        Terrible!|  user3|               No|       0|
|unknown| Canada|         NULL|      NULL|   2020/12/01|   4.0|       I liked it|  user2|                Y|     abc|
|     ??|       |         NULL|   456-def|   12-01-2020|   3.5| 

In [12]:
df.printSchema()

root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- extra: string (nullable = true)
 |    |-- invalid: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- review: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- zip_code: string (nullable = true)



In [29]:
df1=df.withColumn('zip_code',col('zip_code').cast('Integer'))\
      .withColumn('verified_purchase',col('verified_purchase').cast('Boolean'))\
      .withColumn('user_id',lower(col('user_id')))\
      .withColumn('rating',col('rating').cast('Double'))\
      .withColumn('order_date',to_date(col('purchase_date')))\
      .withColumn('metadata_extra',col('metadata').extra)\
      .withColumn('metadata_invalid',col('metadata').invalid)\
      .withColumn('review',trim(col('review')))\
      .drop(col('purchase_date'))\
      .drop(col('metadata'))


df1.show()

+-------+-------+----------+------+-------------+-------+-----------------+--------+----------+--------------+----------------+
|   city|country|product_id|rating|       review|user_id|verified_purchase|zip_code|order_date|metadata_extra|metadata_invalid|
+-------+-------+----------+------+-------------+-------+-----------------+--------+----------+--------------+----------------+
|    n/a|  India|   000-xyz|   5.0|  Best ever!!|  user4|             NULL|    NULL|2021-11-11|          NULL|             999|
|    n/a|   NULL|   123-abc|   5.0|great product|  user1|             true|    NULL|      NULL|         value|            NULL|
|    n/a|   NULL|   123-abc|   5.0|great product|  user1|             true|    NULL|      NULL|         value|            NULL|
|     uk|     UK|   789-ghi|   2.0|    Terrible!|  user3|            false|       0|2022-05-20|          NULL|            NULL|
|unknown| Canada|      NULL|   4.0|   I liked it|  user2|             true|    NULL|      NULL|         

In [33]:
df2=df1.na.drop(subset=['country'])
df2.show()

+-------+-------+----------+------+-----------+-------+-----------------+--------+----------+--------------+----------------+
|   city|country|product_id|rating|     review|user_id|verified_purchase|zip_code|order_date|metadata_extra|metadata_invalid|
+-------+-------+----------+------+-----------+-------+-----------------+--------+----------+--------------+----------------+
|    n/a|  India|   000-xyz|   5.0|Best ever!!|  user4|             NULL|    NULL|2021-11-11|          NULL|             999|
|     uk|     UK|   789-ghi|   2.0|  Terrible!|  user3|            false|       0|2022-05-20|          NULL|            NULL|
|unknown| Canada|      NULL|   4.0| I liked it|  user2|             true|    NULL|      NULL|          NULL|            NULL|
|     ??|       |   456-def|   3.5|   Not bad.|  user2|            false|    NULL|      NULL|          NULL|            NULL|
|       |    USA|          |  NULL|           |   NULL|             NULL|    NULL|      NULL|          NULL|          